# 00B_llava_onevision_7b_snapshot_smoke.ipynb

Generated zero-edit Kaggle runbook. Attach the documented private datasets, keep Internet off, choose the documented accelerator, and click Run All. NON_EVIDENCE_RUNTIME_SMOKE; paper_evidence=false.


In [ ]:
# Generated immutable run identity. There is nothing to edit in this notebook.
import os

STAGE = 'snapshot_smoke'
PROVIDER = 'llava_onevision_7b'
NOTEBOOK_NAME = '00B_llava_onevision_7b_snapshot_smoke.ipynb'
EXPECTED_GPUS = 0
ALLOW_SINGLE_GPU_FALLBACK = True
USE_REAL_MODEL = False
MAX_ITEMS = 2
GLOBAL_SEED = 12013
SCHEMA_VERSION = "certvic.cvpr.output.v2"
SNAPSHOT_CONTRACT = "UNIFIED_SNAPSHOT"
PROMPT_TEMPLATE_ID = "certification_yes_no_v1"
PROMPT_TEMPLATE = "{prompt}\n"
PARSER_VERSION = "certvic.parse.v2"
CANONICAL_RETURN_ZIP = '00B_llava_onevision_7b_snapshot_bundle.zip'
LOCAL_DESTINATION = 'data/runtime/00B_llava_onevision_7b_snapshot_bundle.zip'
INPUT_ROOT = os.environ.get("CERTVIC_KAGGLE_INPUT_ROOT", "/kaggle/input")
WORKING_ROOT = os.environ.get("CERTVIC_KAGGLE_WORKING_ROOT", "/kaggle/working")
SNAPSHOT_DATASET_SLUG = 'certvic/llava-onevision-7b-snapshot'
SNAPSHOT_DATASET_FILENAME = 'llava_onevision_7b_snapshot.zip'
for key, value in {
    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "DIFFUSERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1", "HF_HUB_DISABLE_TELEMETRY": "1", "PIP_NO_INDEX": "1",
    "PIP_DISABLE_PIP_VERSION_CHECK": "1",
}.items():
    os.environ[key] = value


In [ ]:
import hashlib, json, pathlib, shutil, stat, subprocess, sys, zipfile

EARLY_ERRORS = {
    "missing": "KAGGLE_BOOTSTRAP_01_DATASET_NOT_FOUND",
    "ambiguous": "KAGGLE_BOOTSTRAP_02_AMBIGUOUS_DATASET",
    "invalid": "KAGGLE_BOOTSTRAP_03_BUNDLE_INVALID",
    "unsafe": "KAGGLE_BOOTSTRAP_09_UNSAFE_EXTRACTION",
}

def early_sha256(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def early_member(info):
    name = info.filename
    normalized = name.replace("\\", "/")
    value = pathlib.PurePosixPath(normalized)
    mode = (info.external_attr >> 16) & 0xFFFF
    if (not normalized or normalized != name or normalized.endswith("/") or value.is_absolute()
            or ".." in value.parts or "." in value.parts or normalized.startswith("~")
            or "\x00" in normalized or info.is_dir() or stat.S_ISLNK(mode)
            or (mode and not stat.S_ISREG(mode))):
        raise RuntimeError(f"{EARLY_ERRORS['unsafe']}: unsafe member {name!r}")
    return value.as_posix()

def early_verify_and_extract(archive_path, destination):
    source = pathlib.Path(archive_path).resolve()
    try:
        with zipfile.ZipFile(source) as archive:
            infos = archive.infolist()
            names = [early_member(info) for info in infos]
            if len(names) != len(set(names)) or archive.testzip() is not None:
                raise RuntimeError(f"{EARLY_ERRORS['invalid']}: duplicate or corrupt members")
            manifest = json.loads(archive.read("bundle_manifest.json"))
            hashes = json.loads(archive.read("hash_manifest.json"))
            if (manifest.get("schema") != "certvic.kaggle.bundle.v1"
                    or hashes.get("schema") != "certvic.kaggle.hash_manifest.v1"
                    or manifest.get("bundle_type") != 'CODE'
                    or manifest.get("expected_kaggle_dataset_slug") != 'certvic/certvic-code'):
                raise RuntimeError(f"{EARLY_ERRORS['invalid']}: code bundle identity mismatch")
            declared = manifest.get("files", {})
            hash_files = hashes.get("files", {})
            if (set(names) != set(hash_files) | {"hash_manifest.json"}
                    or set(declared) != set(names) - {"bundle_manifest.json", "hash_manifest.json"}):
                raise RuntimeError(f"{EARLY_ERRORS['invalid']}: code file universe mismatch")
            for name, record in hash_files.items():
                payload = archive.read(name)
                observed = {"size": len(payload), "sha256": hashlib.sha256(payload).hexdigest()}
                if record != observed or (name in declared and declared[name] != observed):
                    raise RuntimeError(f"{EARLY_ERRORS['invalid']}: byte mismatch {name}")
            target = pathlib.Path(destination).resolve()
            if target.exists():
                if target.is_symlink() or not target.is_dir():
                    raise RuntimeError(f"{EARLY_ERRORS['unsafe']}: invalid destination")
                shutil.rmtree(target)
            target.mkdir(parents=True)
            for info, name in zip(infos, names, strict=True):
                output = (target / name).resolve()
                try:
                    output.relative_to(target)
                except ValueError as error:
                    raise RuntimeError(f"{EARLY_ERRORS['unsafe']}: traversal member") from error
                output.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as reader, output.open("xb") as writer:
                    shutil.copyfileobj(reader, writer, length=1024 * 1024)
    except (OSError, KeyError, json.JSONDecodeError, zipfile.BadZipFile) as error:
        raise RuntimeError(f"{EARLY_ERRORS['invalid']}: {error}") from error
    return target, manifest

mount = pathlib.Path(INPUT_ROOT) / 'certvic-code'
matches = sorted(path.resolve() for path in mount.rglob('certvic_code_bundle.zip') if path.is_file()) if mount.is_dir() else []
if not matches:
    raise RuntimeError(f"{EARLY_ERRORS['missing']}: slug=certvic/certvic-code filename=certvic_code_bundle.zip")
if len(matches) != 1:
    raise RuntimeError(f"{EARLY_ERRORS['ambiguous']}: slug=certvic/certvic-code candidates={[str(p) for p in matches]}")
CODE_BUNDLE_PATH = str(matches[0])
CODE_BUNDLE_HASH = early_sha256(matches[0])
CODE_EXTRACT_ROOT, CODE_OUTER_MANIFEST = early_verify_and_extract(
    matches[0], pathlib.Path(WORKING_ROOT) / "certvic_code"
)
project_candidates = sorted(path.parent.resolve() for path in CODE_EXTRACT_ROOT.rglob("pyproject.toml")
    if (path.parent / "certvic/__init__.py").is_file())
if len(project_candidates) != 1:
    raise RuntimeError("KAGGLE_BOOTSTRAP_10_AMBIGUOUS_CONTENT: project root")
PROJECT_ROOT = project_candidates[0]
sys.path.insert(0, str(PROJECT_ROOT))

from certvic.cvpr.notebook_bootstrap import (
    discover_unique_file, discover_unique_root, materialize_dataset, verify_attached_bundle,
)
verify_attached_bundle(CODE_BUNDLE_PATH, expected_type='CODE', expected_slug='certvic/certvic-code')
print({"project_root": str(PROJECT_ROOT), "code_bundle_sha256": CODE_BUNDLE_HASH})


In [ ]:
from certvic.cvpr.environment_lock import environment_lock_hash

CONFIG_DATASET = materialize_dataset(
    slug='certvic/certvic-configs', filename='certvic_configs_bundle.zip', expected_type='CONFIGS',
    input_root=INPUT_ROOT, destination=pathlib.Path(WORKING_ROOT) / "certvic_configs",
)
TOOLS_DATASET = materialize_dataset(
    slug='certvic/certvic-execution-tools', filename='certvic_execution_tools_bundle.zip', expected_type='EXECUTION_TOOLS',
    input_root=INPUT_ROOT, destination=pathlib.Path(WORKING_ROOT) / "certvic_execution_tools",
)
WHEELHOUSE_DATASET = materialize_dataset(
    slug='certvic/certvic-offline-wheelhouse', filename='certvic_offline_wheelhouse.zip', expected_type='OFFLINE_LINUX_WHEELHOUSE',
    input_root=INPUT_ROOT, destination=pathlib.Path(WORKING_ROOT) / "certvic_offline_wheelhouse",
)
CONFIG_ROOT = pathlib.Path(CONFIG_DATASET["root"])
WHEELHOUSE_ROOT = pathlib.Path(WHEELHOUSE_DATASET["root"])
ENVIRONMENT_LOCK = str(discover_unique_file(CONFIG_ROOT, "kaggle_t4x2_environment.lock.json"))
ENVIRONMENT_LOCK_HASH = environment_lock_hash(ENVIRONMENT_LOCK)
WHEELHOUSE_MANIFEST = str(discover_unique_file(WHEELHOUSE_ROOT, "wheelhouse_manifest.json"))
WHEELHOUSE_PATH = str(WHEELHOUSE_ROOT / "wheels")
if not pathlib.Path(WHEELHOUSE_PATH).is_dir():
    raise RuntimeError("KAGGLE_BOOTSTRAP_04_WHEELHOUSE_INVALID: wheels directory missing")
MODEL_REGISTRY = str(discover_unique_file(CONFIG_ROOT, "certvic_immutable_model_registry.json"))
ATTACHED_INPUT_HASHES = {
    "code": CODE_BUNDLE_HASH,
    "configs": CONFIG_DATASET["archive_sha256"],
    "tools": TOOLS_DATASET["archive_sha256"],
    "wheelhouse": WHEELHOUSE_DATASET["archive_sha256"],
}
print({"environment_lock": ENVIRONMENT_LOCK, "environment_lock_hash": ENVIRONMENT_LOCK_HASH,
       "wheelhouse_manifest": WHEELHOUSE_MANIFEST, "attached_input_hashes": ATTACHED_INPUT_HASHES})


In [ ]:
from certvic.cvpr.model_snapshot_manifest import verify_manifest

SNAPSHOT_DATASET = materialize_dataset(
    slug='certvic/llava-onevision-7b-snapshot', filename='llava_onevision_7b_snapshot.zip', expected_type="MODEL_SNAPSHOT",
    input_root=INPUT_ROOT,
    destination=pathlib.Path(WORKING_ROOT) / 'certvic_snapshot_llava_onevision_7b',
)
SNAPSHOT_CONTAINER = pathlib.Path(SNAPSHOT_DATASET["root"])
SNAPSHOT_MANIFEST = str(discover_unique_file(
    SNAPSHOT_CONTAINER, "certvic_model_snapshot_manifest.json"
))
SNAPSHOT_ROOT = pathlib.Path(SNAPSHOT_MANIFEST).parent.resolve()
MODEL_PATH = str(SNAPSHOT_ROOT)
PROCESSOR_PATH = str(SNAPSHOT_ROOT)
SNAPSHOT_MANIFEST_HASH = early_sha256(SNAPSHOT_MANIFEST)
snapshot_identity = json.loads(pathlib.Path(SNAPSHOT_MANIFEST).read_text(encoding="utf-8"))
MODEL_ID = str(snapshot_identity["model_id"])
PROCESSOR_ID = str(snapshot_identity["processor_id"])
MODEL_COMMIT = str(snapshot_identity["model_commit"])
PROCESSOR_COMMIT = str(snapshot_identity["processor_commit"])
EXPECTED_ARCHITECTURE = str(snapshot_identity["expected_architecture"])
SNAPSHOT_ROOT_HASH = str(snapshot_identity["unified_snapshot_root_sha256"])
outer_snapshot = SNAPSHOT_DATASET["bundle_manifest"]
for field, expected in {
    "provider": PROVIDER, "model_id": MODEL_ID, "model_commit": MODEL_COMMIT,
    "processor_commit": PROCESSOR_COMMIT, "expected_architecture": EXPECTED_ARCHITECTURE,
    "unified_snapshot_root_sha256": SNAPSHOT_ROOT_HASH,
}.items():
    if outer_snapshot.get(field) != expected:
        raise RuntimeError(f"KAGGLE_BOOTSTRAP_03_BUNDLE_INVALID: snapshot {field} mismatch")
registry = json.loads(pathlib.Path(MODEL_REGISTRY).read_text(encoding="utf-8"))["models"][PROVIDER]
if (registry.get("repository_id") != MODEL_ID
        or registry.get("model_commit") != MODEL_COMMIT
        or registry.get("processor_commit") != PROCESSOR_COMMIT
        or registry.get("architecture") != EXPECTED_ARCHITECTURE):
    raise RuntimeError("KAGGLE_BOOTSTRAP_08_RUN_IDENTITY_INCOMPLETE: immutable registry mismatch")
ATTACHED_INPUT_HASHES["snapshot"] = SNAPSHOT_DATASET["archive_sha256"]
print({"snapshot_root": MODEL_PATH, "snapshot_manifest": SNAPSHOT_MANIFEST,
       "snapshot_manifest_sha256": SNAPSHOT_MANIFEST_HASH,
       "snapshot_root_sha256": SNAPSHOT_ROOT_HASH, "model_id": MODEL_ID,
       "model_commit": MODEL_COMMIT, "processor_commit": PROCESSOR_COMMIT,
       "expected_architecture": EXPECTED_ARCHITECTURE})


In [ ]:
from certvic.cvpr.environment_lock import (
    offline_environment_flags, prepare_offline_environment,
)
from certvic.cvpr.notebook_bootstrap import configure_offline_environment, import_smoke
from certvic.cvpr.runtime_preflight import hardware_report

configure_offline_environment()
if offline_environment_flags().get("HF_HUB_OFFLINE") != "1" or os.environ.get("PIP_NO_INDEX") != "1":
    raise RuntimeError("KAGGLE_ZERO_EDIT_OFFLINE_POLICY_INCOMPLETE")
environment_verification = prepare_offline_environment(
    ENVIRONMENT_LOCK,
    wheelhouse=WHEELHOUSE_PATH,
    wheelhouse_manifest=WHEELHOUSE_MANIFEST,
    allow_preinstalled=True,
    require_exact=True,
    require_cuda=False,
)
if environment_verification["status"] not in {
    "EXACT_PREINSTALLED_ENVIRONMENT_ACCEPTED", "OFFLINE_WHEELHOUSE_INSTALLED_AND_VERIFIED",
}:
    raise RuntimeError("KAGGLE_ZERO_EDIT_EXACT_ENVIRONMENT_NOT_ESTABLISHED")
hardware = hardware_report()
print(hardware)
if EXPECTED_GPUS == 0 and (hardware["cuda_available"] or hardware["gpu_count"] != 0):
    raise RuntimeError("KAGGLE_ZERO_EDIT_CPU_ACCELERATOR_MUST_BE_OFF")
if EXPECTED_GPUS > 0:
    names = [row["name"] for row in hardware.get("gpus", [])]
    if not hardware["cuda_available"]:
        raise RuntimeError("KAGGLE_BOOTSTRAP_07_GPU_CONTRACT_FAILED: CUDA unavailable")
    if len(names) < 2 and not (len(names) == 1 and ALLOW_SINGLE_GPU_FALLBACK):
        raise RuntimeError(f"KAGGLE_BOOTSTRAP_07_GPU_CONTRACT_FAILED: device_count={len(names)}")
    if not all("T4" in name.upper() for name in names[:2]):
        raise RuntimeError(f"KAGGLE_BOOTSTRAP_07_GPU_CONTRACT_FAILED: devices={names}")
import_versions = import_smoke(["certvic", "numpy", "pandas", "torch", "transformers"])
print({"offline": True, "network_used": False, "imports": import_versions,
       "environment_status": environment_verification["status"]})


In [ ]:
from certvic.cvpr.smoke_artifacts import write_snapshot_artifacts

snapshot = verify_manifest(
    MODEL_PATH, SNAPSHOT_MANIFEST,
    expected_model_id=MODEL_ID,
    expected_model_commit=MODEL_COMMIT,
    expected_processor_commit=PROCESSOR_COMMIT,
    expected_architecture=EXPECTED_ARCHITECTURE,
)
if not snapshot["passed"]:
    raise RuntimeError(f"KAGGLE_ZERO_EDIT_SNAPSHOT_INVALID: {snapshot['errors']}")
artifact = write_snapshot_artifacts(WORKING_ROOT, PROVIDER, {
    **snapshot,
    "snapshot_contract": SNAPSHOT_CONTRACT,
    "model_id": MODEL_ID,
    "processor_id": PROCESSOR_ID,
    "model_commit": MODEL_COMMIT,
    "processor_commit": PROCESSOR_COMMIT,
    "expected_architecture": EXPECTED_ARCHITECTURE,
    "snapshot_manifest_file_sha256": SNAPSHOT_MANIFEST_HASH,
    "snapshot_root_hash": SNAPSHOT_ROOT_HASH,
    "snapshot_archive_sha256": SNAPSHOT_DATASET["archive_sha256"],
})
canonical = pathlib.Path(WORKING_ROOT) / CANONICAL_RETURN_ZIP
if not canonical.is_file():
    raise RuntimeError(f"KAGGLE_ZERO_EDIT_CANONICAL_RETURN_MISSING: {CANONICAL_RETURN_ZIP}")
print(str(canonical))
print(f"DOWNLOAD_FILENAME={CANONICAL_RETURN_ZIP}")
print(f"LOCAL_DESTINATION={LOCAL_DESTINATION}")
print("RESUME_COMMAND=python3 scripts/run_all_cpu_workflows.py --resume")
